# Recording Data Provenance with ``aiida-gromacs``

This tutorial follows the first six steps of Justin Lemkul’s [lysozyme tutorial](http://www.mdtutorials.com/gmx/lysozyme/). We will not explain each individual step as this can be found on Justin’s webpage, but we will link to each page and show the ``aiida-gromacs`` equivalant command.

<center><img src="lysozyme_files/images/lemkul.png" alt="Lysozyme structure" width="50%" /></center>

Please also note the slight differences in commands between the tutorial and that by Justin Lemkul is simply down to the way we are recording provenance, which requires non-interactive input into the gromacs tools.

First, we will create an AiiDA profile, start the daemon and check the status of the aiida services using ``verdi`` (the command line interface utility for AiiDA).

In [ ]:
! verdi presto --use-zeromq
! verdi daemon start 2
! verdi status

At each of the below steps you should run ``verdi`` to view the status of the submitted process before moving onto the next step, you do this by:

In [ ]:
! verdi process list -a

In this notebook AiiDA, the aiida-gromacs plugin and dependent tools are pre-installed. Here's a brief description of the tools used; 
* AiiDA uses a [PostgreSQL](https://www.postgresql.org) database to store all data produced and the links between input and output files for each command run. Each submitted command is termed a process in AiiDA. 
* Communication between submitted processes are handled with [RabbitMQ](https://www.rabbitmq.com/) and submitted processes are handled with a deamon process that runs in the background. 
* ``aiida-gromacs`` requires an installation of [GROMACS](https://www.gromacs.org/) and the path to where it is installed.

1. We will start from the [pbd2gmx](http://www.mdtutorials.com/gmx/lysozyme/01_pdb2gmx.html)  step of Justin’s tutorial, with the ``aiida-gromacs`` equivalent python wrapper ``gmx_pdb2gmx``:

In [ ]:
! gmx_pdb2gmx -f lysozyme_files/inputs/1AKI_clean.pdb -ff oplsaa -water spce -o 1AKI_forcefield.gro -p 1AKI_topology.top -i 1AKI_restraints.itp

A successfully finished process will exit with code [0]. 

In [ ]:
! verdi process list -a

2. & 3. Next we will create the [box and then solvate](http://www.mdtutorials.com/gmx/lysozyme/03_solvate.html):

In [ ]:
! gmx_editconf -f 1AKI_forcefield.gro -center 0 -d 1.0 -bt cubic -o 1AKI_newbox.gro

In [ ]:
! verdi process list -a

In [ ]:
! gmx_solvate -cp 1AKI_newbox.gro -cs spc216.gro -p 1AKI_topology.top -o 1AKI_solvated.gro

In [ ]:
! verdi process list -a

4. & 5. Then add [ions](http://www.mdtutorials.com/gmx/lysozyme/04_ions.html) to neutralise the system after preprocessing the topology:

In [ ]:
! gmx_grompp -f lysozyme_files/inputs/ions.mdp -c 1AKI_solvated.gro -p 1AKI_topology.top -o 1AKI_ions.tpr

In [ ]:
! verdi process list -a

In [ ]:
! gmx_genion -s 1AKI_ions.tpr -p 1AKI_topology.top -pname NA -nname CL -neutral true -o 1AKI_solvated_ions.gro

In [ ]:
! verdi process list -a

6. & 7. Then [minimise](http://www.mdtutorials.com/gmx/lysozyme/05_EM.html) the system after preprocessing the topology:

In [ ]:
! gmx_grompp -f lysozyme_files/inputs/min.mdp -c 1AKI_solvated_ions.gro -p 1AKI_topology.top -o 1AKI_minimised.tpr

In [ ]:
! verdi process list -a

In [ ]:
! gmx_mdrun -s 1AKI_minimised.tpr -c 1AKI_minimised.gro -e 1AKI_minimised.edr -g 1AKI_minimised.log -o 1AKI_minimised.trr

In [ ]:
! verdi process list -a

We can view the provenance graph of these processes, which shows how inputs and outputs of each process are connected to other processes. To save the provenance graph of all finished processes, replace the primary key value <PK> in the command below with that of the most recently run process.

In [ ]:
! verdi node graph generate 54

The graph should look something like this:

<center><img src="lysozyme_files/images/54.dot.png" alt="Lysozyme structure" width="90%" /></center>

To view all the currently run commands in each process, the input files used in each command and the output files produced from each command, we can use:

In [ ]:
! verdi data gromacs.provenance show

At the end of a project, the AiiDA database can be saved as an AiiDA archive file (sqlite/zip format) for long term storage and to share your data and provenance with others. 

In [ ]:
! verdi archive create --all lysozyme_tutorial.aiida

We hope to share further tutorials on loading, querying and displaying data from AiiDA archives. Watch this space!